In [1]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path('/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd') 
image_dir = data_dir / 'images/Images'


for item in data_dir.rglob('*'):
    print(item)

/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/Imagewise_Data.csv
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/Annotation.json
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/Patientwise_Data.csv
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/N-270-06.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/R-121-05.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/N-221-01.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/N-262-05.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/N-353-08.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/S-157-03.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/Images/R-62-02.jpg
/kaggle/input/datasets/syedjaffarrazakazmi/oca-and-opmd/images/

In [2]:
with open(data_dir / 'Annotation.json') as f: 
    coco = json.load(f)

print("Top-level keys:", coco.keys())
print("Num images:", len(coco['images']))
print("Num annotations:", len(coco['annotations']))
print("Categories:", coco['categories'])

Top-level keys: dict_keys(['images', 'annotations', 'categories'])
Num images: 3000
Num annotations: 6358
Categories: [{'id': 1, 'name': 'Lesion', 'supercategory': 'object'}, {'id': 2, 'name': 'Oral Cavity', 'supercategory': 'object'}]


In [3]:
df_imagewise = pd.read_csv(data_dir / 'Imagewise_Data.csv')
df_patientwise = pd.read_csv(data_dir / 'Patientwise_Data.csv')

print(df_imagewise.shape)
print(df_patientwise.shape)

print(df_imagewise.head())
print(df_patientwise.head())


(3000, 4)
(714, 7)
  Image Name Category Clinical Diagnosis  Lesion Annotation Count
0    R-01-01     OPMD        Leukoplakia                        1
1    R-01-02     OPMD        Leukoplakia                        1
2    R-01-03   Benign      Coated Tongue                        1
3    R-02-01   Benign                VBD                        1
4    R-02-02   Benign                VBD                        1
  Patient ID  Age Gender Smoking Chewing_Betel_Quid  Alcohol  Image Count
0       R-01   63      M      No                  No      No            3
1       R-02   17      F      No                  No      No            8
2       R-03   70      M      No                  No      No            5
3       R-04   45      M      No                  No      No            5
4       R-05   46      M      No                 Yes      No            2


In [4]:
coco_image_ids = set(img['id'] for img in coco['images'])
coco_filenames = set(img['file_name'] for img in coco['images'])

csv_filenames = set(df_imagewise['Image Name'])

print("In COCO but not in CSV:", len(coco_filenames - csv_filenames))
print("In CSV but not in COCO:", len(csv_filenames - coco_filenames))

In COCO but not in CSV: 3000
In CSV but not in COCO: 3000


In [5]:
# Look at actual raw values
print("COCO file_name sample:", [img['file_name'] for img in coco['images'][:5]])
print("CSV Image Name sample:", df_imagewise['Image Name'].head(5).tolist())

COCO file_name sample: ['R-01-01.jpg', 'R-01-02.jpg', 'R-01-03.jpg', 'R-02-01.jpg', 'R-02-02.jpg']
CSV Image Name sample: ['R-01-01', 'R-01-02', 'R-01-03', 'R-02-01', 'R-02-02']


In [6]:
coco_filenames_stripped = set(Path(img['file_name']).stem for img in coco['images'])
csv_filenames = set(df_imagewise['Image Name'])

print("In COCO but not in CSV:", len(coco_filenames_stripped - csv_filenames))
print("In CSV but not in COCO:", len(csv_filenames - coco_filenames_stripped))

In COCO but not in CSV: 0
In CSV but not in COCO: 0


In [7]:
image_ids_in_images = set(img['id'] for img in coco['images'])
image_ids_in_annotations = set(a['id'] for a in coco['annotations'])

orphaned_annotations = image_ids_in_annotations - image_ids_in_images
print("Orphaned annotations (no matching image):", len(orphaned_annotations))

Orphaned annotations (no matching image): 3358


In [8]:
# Look at actual raw values
print("Sample image IDs in 'images':", [img['id'] for img in coco['images'][:5]])
print("Sample image_ids in 'annotations':", [a['image_id'] for a in coco['annotations'][:5]])

# Are these the same type? int vs str mismatch is a classic cause
print(type(coco['images'][0]['id']), type(coco['annotations'][0]['image_id']))

Sample image IDs in 'images': [1, 2, 3, 4, 5]
Sample image_ids in 'annotations': [1, 1, 2, 2, 3]
<class 'int'> <class 'int'>


In [9]:
# Find the keys actually present across all image entries
all_keys = set()
for img in coco['images']:
    all_keys.update(img.keys())
print("All keys seen across image entries:", all_keys)

# Find which entries are missing 'width'
missing_width = [img for img in coco['images'] if 'width' not in img]
print("Entries missing 'width':", len(missing_width))
print("Sample malformed entry:", missing_width[0] if missing_width else None)

All keys seen across image entries: {'id', 'file_name'}
Entries missing 'width': 3000
Sample malformed entry: {'id': 1, 'file_name': 'R-01-01.jpg'}


In [10]:
img_ids = sorted(img['id'] for img in coco['images'])
ann_img_ids = sorted(set(a['image_id'] for a in coco['annotations']))

print("Image ID range:", min(img_ids), "to", max(img_ids), "| count:", len(img_ids))
print("Annotation image_id range:", min(ann_img_ids), "to", max(ann_img_ids), "| count of unique:", len(ann_img_ids))

# How many annotation image_ids actually exist in images?
valid = set(img_ids) & set(ann_img_ids)
print("Annotation image_ids that DO exist in images:", len(valid), "out of", len(ann_img_ids), "unique ids referenced")

Image ID range: 1 to 3000 | count: 3000
Annotation image_id range: 1 to 3000 | count of unique: 3000
Annotation image_ids that DO exist in images: 3000 out of 3000 unique ids referenced


In [11]:
image_ids_in_images = set(img['id'] for img in coco['images'])
image_ids_in_annotations = set(a['image_id'] for a in coco['annotations'])

orphaned_annotations = image_ids_in_annotations - image_ids_in_images

print("Unique orphaned image_ids:", len(orphaned_annotations))
print("Sample orphaned IDs:", list(orphaned_annotations)[:10])
print("Max valid image id:", max(image_ids_in_images))

Unique orphaned image_ids: 0
Sample orphaned IDs: []
Max valid image id: 3000


In [12]:
image_ids_in_images = set(img['id'] for img in coco['images'])
image_ids_in_annotations = set(a['id'] for a in coco['annotations'])

orphaned_annotations = image_ids_in_annotations - image_ids_in_images
print("Orphaned annotations (no matching image):", len(orphaned_annotations))

Orphaned annotations (no matching image): 3358


In [13]:
from PIL import Image

dims_cache = {}
missing_files = []


for img in coco['images']:
    img_path = image_dir / img['file_name']
    if img_path.exists():
        with Image.open(img_path) as im:
            dims_cache[img['id']] = im.size
    else:
        missing_files.append(img['file_name'])

In [14]:
bad_boxes = []

for ann in coco['annotations']:
    w, h = dims_cache[ann['image_id']]
    x, y, bw, bh = ann['bbox']
    if x < 0 or y < 0 or x + bw > w or y + bh > h:
        bad_boxes.append(ann['id'])

print("Out-of-bounds bboxes:", len(bad_boxes))

Out-of-bounds bboxes: 143


In [15]:
bad_ann_ids = set()
bad_image_ids = set()

for ann in coco['annotations']:
    w, h = dims_cache[ann['image_id']]
    x, y, bw, bh = ann['bbox']
    if x < 0 or y < 0 or x + bw > w or y + bh > h:
        bad_ann_ids.add(ann['id'])
        bad_image_ids.add(ann['image_id'])

print("Bad annotations:", len(bad_ann_ids))
print("Unique images affected:", len(bad_image_ids))

violations = []
for ann in coco['annotations']:
    w, h = dims_cache[ann['image_id']]
    x, y, bw, bh = ann['bbox']
    overflow_x = max(0, (x + bw) - w)
    overflow_y = max(0, (y + bh) - h)
    if x < 0 or y < 0 or overflow_x > 0 or overflow_y > 0:
        violations.append({
            'ann_id': ann['id'],
            'image_id': ann['image_id'],
            'overflow_x': overflow_x,
            'overflow_y': overflow_y,
            'neg_x': min(0, x),
            'neg_y': min(0, y),
        })

vdf = pd.DataFrame(violations)
print("\nOverflow stats (pixels):")
print(vdf[['overflow_x', 'overflow_y', 'neg_x', 'neg_y']].describe())

Bad annotations: 143
Unique images affected: 139

Overflow stats (pixels):
       overflow_x   overflow_y  neg_x  neg_y
count  143.000000   143.000000  143.0  143.0
mean     3.727273  1183.664336    0.0    0.0
std     44.571699   556.154160    0.0    0.0
min      0.000000     0.000000    0.0    0.0
25%      0.000000   746.500000    0.0    0.0
50%      0.000000  1238.000000    0.0    0.0
75%      0.000000  1553.500000    0.0    0.0
max    533.000000  2185.000000    0.0    0.0


In [16]:
# Check dimensions of affected vs clean images
affected_ids = set(v['image_id'] for v in violations)
clean_ids = set(img['id'] for img in coco['images']) - affected_ids

affected_dims = [dims_cache[i] for i in affected_ids]
clean_dims = [dims_cache[i] for i in clean_ids]

affected_df = pd.DataFrame(affected_dims, columns=['width', 'height'])
clean_df = pd.DataFrame(clean_dims, columns=['width', 'height'])

print("Affected images — width/height stats:")
print(affected_df.describe())
print("\nClean images — width/height stats:")
print(clean_df.describe())

Affected images — width/height stats:
             width       height
count   139.000000   139.000000
mean   4000.633094  1875.654676
std     138.591054   317.443678
min    3000.000000  1800.000000
25%    4000.000000  1800.000000
50%    4000.000000  1800.000000
75%    4000.000000  1800.000000
max    4608.000000  4000.000000

Clean images — width/height stats:
             width       height
count  2861.000000  2861.000000
mean   3796.929745  3150.847955
std     730.862628   678.612372
min     654.000000   597.000000
25%    3264.000000  2448.000000
50%    4000.000000  3264.000000
75%    4608.000000  3456.000000
max    4608.000000  4608.000000


In [17]:
# Find the worst offenders
worst = sorted(violations, key=lambda v: v['overflow_y'], reverse=True)[:5]
for v in worst:
    img_id = v['image_id']
    w, h = dims_cache[img_id]
    ann = next(a for a in coco['annotations'] if a['id'] == v['ann_id'])
    print(f"image_id {img_id} | image size: {w}x{h} | bbox: {ann['bbox']} | overflow_y: {v['overflow_y']:.0f}px")

image_id 1403 | image size: 4000x1800 | bbox: [20, 584, 1746, 3401] | overflow_y: 2185px
image_id 1417 | image size: 4000x1800 | bbox: [20, 519, 1706, 3466] | overflow_y: 2185px
image_id 1307 | image size: 4000x1800 | bbox: [13, 886, 1773, 3092] | overflow_y: 2178px
image_id 1389 | image size: 4000x1800 | bbox: [20, 702, 1766, 3276] | overflow_y: 2178px
image_id 1317 | image size: 4000x1800 | bbox: [26, 1260, 1760, 2712] | overflow_y: 2172px


In [18]:
# For affected images, check if swapping w/h would make bboxes valid
salvageable = []
truly_broken = []

for v in violations:
    img_id = v['image_id']
    w, h = dims_cache[img_id]
    ann = next(a for a in coco['annotations'] if a['id'] == v['ann_id'])
    x, y, bw, bh = ann['bbox']
    
    # Would this bbox be valid if image were rotated (w and h swapped)?
    if x >= 0 and y >= 0 and x + bw <= h and y + bh <= w:
        salvageable.append(img_id)
    else:
        truly_broken.append(img_id)

print("Salvageable (rotation fix):", len(set(salvageable)))
print("Truly broken:", len(set(truly_broken)))

Salvageable (rotation fix): 139
Truly broken: 0


In [19]:
import shutil

# Set up output directory
output_image_dir = Path('/kaggle/working/piyarathne/images')
output_image_dir.mkdir(parents=True, exist_ok=True)

rotated_count = 0
copied_count = 0

for img in coco['images']:
    src_path = image_dir / img['file_name']
    dst_path = output_image_dir / img['file_name']
    
    if img['id'] in affected_ids:
        with Image.open(src_path) as im:
            rotated = im.rotate(90, expand=True)
            rotated.save(dst_path)
            dims_cache[img['id']] = rotated.size
        rotated_count += 1
    else:
        shutil.copy2(src_path, dst_path)
        copied_count += 1

print("Rotated:", rotated_count)
print("Copied:", copied_count)
print("Total:", rotated_count + copied_count)

Rotated: 139
Copied: 2861
Total: 3000


In [20]:
bad_boxes_after = []
for ann in coco['annotations']:
    w, h = dims_cache[ann['image_id']]
    x, y, bw, bh = ann['bbox']
    if x < 0 or y < 0 or x + bw > w or y + bh > h:
        bad_boxes_after.append(ann['id'])

print("Out-of-bounds bboxes after fix:", len(bad_boxes_after))

Out-of-bounds bboxes after fix: 0


In [21]:
# Step 6 — Class distribution
cat_id_to_name = {c['id']: c['name'] for c in coco['categories']}

ann_df = pd.DataFrame(coco['annotations'])
ann_df['category_name'] = ann_df['category_id'].map(cat_id_to_name)

print("COCO detection categories:\n", ann_df['category_name'].value_counts())
print("\nClinical Category:\n", df_imagewise['Category'].value_counts())
print("\nClinical Diagnosis:\n", df_imagewise['Clinical Diagnosis'].value_counts())

COCO detection categories:
 category_name
Lesion         3358
Oral Cavity    3000
Name: count, dtype: int64

Clinical Category:
 Category
OPMD       1394
Benign      748
Healthy     729
OCA         129
Name: count, dtype: int64

Clinical Diagnosis:
 Clinical Diagnosis
Normal Mucosa         728
OSF                   684
OLP                   358
Oral Cancer           125
VBD                   123
                     ... 
OSF+Cyst                1
Allergic Reaction       1
Petichiae               1
Mucosil                 1
Alevolar Keratosis      1
Name: count, Length: 155, dtype: int64


In [22]:
from pathlib import Path

records = []
for img in coco['images']:
    img_id = img['id']
    file_stem = Path(img['file_name']).stem
    anns = [a for a in coco['annotations'] if a['image_id'] == img_id]
    coco_categories = [cat_id_to_name[a['category_id']] for a in anns]
    w, h = dims_cache[img_id] 

    records.append({
        'source_dataset': 'piyarathne',
        'image_id': img_id,
        'file_name': img['file_name'],
        'image_stem': file_stem,
        'width': w,
        'height': h,
        'coco_categories': coco_categories,
    })

standardized_df = pd.DataFrame(records)

# Merge clinical data
standardized_df = standardized_df.merge(
    df_imagewise[['Image Name', 'Category', 'Clinical Diagnosis', 'Lesion Annotation Count']],
    left_on='image_stem',
    right_on='Image Name',
    how='left'
)

# Extract patient ID
standardized_df['patient_id'] = standardized_df['image_stem'].str.extract(r'^([A-Z]-\d+)-\d+$')

# For C prefix (4 segments), override with first two parts
c_mask = standardized_df['image_stem'].str.match(r'^C-\d+-\d+-\d+$')
standardized_df.loc[c_mask, 'patient_id'] = standardized_df.loc[c_mask, 'image_stem'].str.extract(r'^(C-\d+)-\d+-\d+$')[0]

# print("Unmatched patient IDs after fix:", standardized_df['patient_id'].isna().sum())
print("Unique patients found:", standardized_df['patient_id'].nunique())
print("Patients in patientwise CSV:", df_patientwise['Patient ID'].nunique())

Unique patients found: 714
Patients in patientwise CSV: 714


In [23]:
# Step 7 — Patient-level distribution
patient_img_counts = standardized_df.groupby('patient_id').size()
print("Images per patient — min/max/median:",
      patient_img_counts.min(), patient_img_counts.max(), patient_img_counts.median())
print("\nPatients per Clinical Diagnosis:")
print(standardized_df.groupby('Clinical Diagnosis')['patient_id'].nunique())

# Step 8 — Resolution (rotation-corrected)
widths  = standardized_df['width'].tolist()
heights = standardized_df['height'].tolist()
print("\nWidth  — min/max/median:", min(widths), max(widths), pd.Series(widths).median())
print("Height — min/max/median:", min(heights), max(heights), pd.Series(heights).median())

Images per patient — min/max/median: 1 18 4.0

Patients per Clinical Diagnosis:
Clinical Diagnosis
Alevolar Keratosis          1
Allergic Reaction           1
Allergy                     3
Alveolar Keratosis          2
Alveolar Kertitis           1
                           ..
Verucouss Ca                1
Vesiculo Bullous Disease    1
Viral Infection/ RAU        1
Viral Wart                  2
White Coated Tongue         1
Name: patient_id, Length: 155, dtype: int64

Width  — min/max/median: 654 4608 4000.0
Height — min/max/median: 597 4608 3264.0


In [24]:
# Step 9 — Export
standardized_df.to_csv('/kaggle/working/piyarathne_standardized.csv', index=False)
print("Exported:", standardized_df.shape)

# Step 10 — Final sanity check
assert standardized_df['image_id'].nunique() == 3000, "Row count mismatch"
assert standardized_df['Clinical Diagnosis'].isna().sum() == 0, "Unmatched clinical data"
assert standardized_df['patient_id'].isna().sum() == 0, "Unmatched patient IDs"
print("All assertions passed")
print(standardized_df.columns.tolist())
standardized_df.head(5)

Exported: (3000, 12)
All assertions passed
['source_dataset', 'image_id', 'file_name', 'image_stem', 'width', 'height', 'coco_categories', 'Image Name', 'Category', 'Clinical Diagnosis', 'Lesion Annotation Count', 'patient_id']


,source_dataset,image_id,file_name,image_stem,width,height,coco_categories,Image Name,Category,Clinical Diagnosis,Lesion Annotation Count,patient_id
0,piyarathne,1,R-01-01.jpg,R-01-01,4608,3456,"[Lesion, Oral Cavity]",R-01-01,OPMD,Leukoplakia,1,R-01
1,piyarathne,2,R-01-02.jpg,R-01-02,4608,3456,"[Oral Cavity, Lesion]",R-01-02,OPMD,Leukoplakia,1,R-01
2,piyarathne,3,R-01-03.jpg,R-01-03,3456,4608,"[Oral Cavity, Lesion]",R-01-03,Benign,Coated Tongue,1,R-01
3,piyarathne,4,R-02-01.jpg,R-02-01,4608,3456,"[Lesion, Oral Cavity]",R-02-01,Benign,VBD,1,R-02
4,piyarathne,5,R-02-02.jpg,R-02-02,4608,3456,"[Lesion, Oral Cavity]",R-02-02,Benign,VBD,1,R-02
